# 01 · Understanding Demonstrations
**Time: 20 minutes.** Select the **Manipulation Lab (Python 3.12)** kernel, then run each cell with Shift+Enter.

## Learning Objectives
Identify observations and actions, inspect a complete human demonstration, and explain the fixed episode split.

## Background
A demonstration is a recorded sequence of observations and expert actions. An observation describes the current robot and object state; an action commands the next control step. Behavior cloning later learns this mapping. Demonstration playback restores recorded states; it is not a learned policy rollout.

## Step-by-step Instructions
### Step 1 — Start your group's session
The instructor sets `LAB_GROUP`. This cell creates a new versioned run; rerunning it starts a new assignment session and preserves previous results. Notebooks 02–04 automatically reopen this group's latest session. Restarting only their kernels does not create a new session.


In [ ]:
from pathlib import Path
import os, json
from IPython.display import display, Markdown, Image, Video
from manipulation_lab.common import ROOT, DATA, write_json
from manipulation_lab.notebook import workspace, command, next_stage, stage_path
from manipulation_lab.teaching import read_json, markdown_table
GROUP = os.environ.get("LAB_GROUP", "group_01")  # Use the group assigned by your instructor.
RUN = workspace(GROUP, fresh=True)
print("Dataset:", DATA)
if not DATA.is_file():
    raise FileNotFoundError("Dataset missing. From the repository root run: scripts/lab_python.sh scripts/download_dataset.py")


### Step 2 — Inspect the data contract
Expect 200 episodes and 23,207 control steps, split into 160 training, 20 validation and 20 test episodes. Entire episodes stay in one split. Only training observations determine normalization.

The policy receives **16 values**: end-effector position (3), orientation (4), gripper joint positions (2), and can pose (7). Positions are world-frame meters, with Z upward; quaternions use xyzw order. The dataset's 14-value object vector also contains relative sensors; the fixed adapter uses `object[7:14]` consistently during training and inference.

The **7 normalized actions** are three world translation increments, three axis-angle rotation increments, and one gripper command. Translation/rotation limits map to ±0.05 m / ±0.5 rad. Positive gripper commands close; negative commands open. The control frequency is 20 Hz; these indices are not acquisition timestamps.


In [ ]:
from manipulation_lab.dataset import inspect
import h5py, numpy as np
info = inspect(DATA)
print("Episodes:", info["episode_count"], "Control steps:", info["total_steps"])
print("Raw observations:", info["observation_shapes"])
print("Policy observations:", info["policy_observation_shapes"])
print("Split sizes:", {k: len(v) for k, v in info["splits"].items()})
print(info["time"])


### Step 3 — Play and inspect one demonstration
Choose `EPISODE` from `demo_0` to `demo_199`. Record one example of how the gripper command relates to object motion. The scatter plot shows the **first state of each demonstration**, not every trajectory state.


In [ ]:
EPISODE = "demo_0"
with h5py.File(DATA, "r") as f:
    d = f["data"][EPISODE]
    expert_actions = d["actions"][:]
    world_position = d["obs/object"][:, 7:10]
    eef_position = d["obs/robot0_eef_pos"][:]
    for step in [0, len(expert_actions) // 2, len(expert_actions) - 1]:
        print("Step", step, "end effector (world m):", eef_position[step],
              "can (world m):", world_position[step], "normalized action:", expert_actions[step])
DEMO = next_stage(RUN, "demonstration")
command("replay_demo.py", "--dataset", DATA, "--episode", EPISODE, "--output", DEMO)
from manipulation_lab.visualization import demonstration_figures
demonstration_figures(info, EPISODE, world_position, expert_actions, DEMO)
write_json(DEMO / "dataset_inspection.json", info)
display(Video(str(DEMO / "demonstration.mp4"), embed=True))
for filename in ["demonstration_signals.png", "demonstration_positions.png"]:
    display(Image(filename=str(DEMO / filename)))
print("Saved video, figures and data inspection:", DEMO)


## Expected Outputs
A real demonstration video, separate position and action plots, a split-specific initial-position plot, and a dataset inspection JSON. All are saved in your run directory.

## Questions for Students
**Q1. What observations and actions are used in this task?** State the coordinate frame, dimensions and units. Also explain why full trajectories are more informative than successful final images. Fill the strings below in your own words and rerun this cell to save them.


In [ ]:
Q1 = ""
TRAJECTORY_REFLECTION = ""
write_json(RUN / "demonstration_analysis.json", {
    "episode": EPISODE, "length": len(expert_actions), "source": "human_demonstration",
    "observations": info["policy_observation_shapes"], "action_dim": 7,
    "q1": Q1, "q2": TRAJECTORY_REFLECTION,
    "answers_complete": bool(Q1.strip() and TRAJECTORY_REFLECTION.strip()),
    "dataset_sha256": info["dataset_sha256"]})
print("Saved answers. Blank answers keep the final report in DRAFT status.")


## Completion Checklist
- [ ] I played a recorded human demonstration and identified the object interaction.
- [ ] I can distinguish observations, normalized commands and physical positions.
- [ ] I inspected the fixed split and initial-position distribution.
- [ ] I saved Q1 and the trajectory reflection. Continue to Notebook 02 using the same group.
